# Stelow cohort — lymphocyte panel (Panel 3)

**Goal:** Characterize immune cell infiltration in MHC II–high vs MHC II–low
LUAD tumors using multiplexed immunofluorescence (Panel 3: DAPI, CD3, CD56,
CD8, MHC I, MHC II, PanCK) from large tissue sections. Cell densities are
normalized by tissue area (cells/mm²) per region (tumor, stroma, alveoli)
and compared by Mann-Whitney U test with Benjamini-Hochberg FDR correction.

**Input:** Panel 3 HALO summary CSV (one row per ROI), Stelow cohort metadata CSV.

**Output:** Figure 5C (PanCK+MHCII+/- stacked bar), Figure 5D (lymphocyte
density by region); supplementary tables.

In [ ]:
from pathlib import Path
import yaml
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])
fig_dir   = repo_root / cfg['outputs']['figures']
table_dir = repo_root / cfg['outputs']['tables']

fig_out   = fig_dir   / 'figure5'
supp_out  = fig_dir   / 'figure5-supp'
table_out = table_dir / 'figure5'

fig_out.mkdir(parents=True, exist_ok=True)
supp_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

In [ ]:
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']

custom_palette = {
    'class II high': cmap_high_low[0],
    'class II low':  cmap_high_low[1],
}

## Load data

Panel 3 HALO summary CSV contains one row per ROI with cell counts and tissue
area per classifier region (tumor, stroma, alveoli). Patient and ROI identifiers
are extracted from the image location string. Stelow cohort clinical metadata
is merged on patient code.

In [ ]:
panel3_path  = Path('/project/Dolatshahi_Lab/Gabe_Hanson/data/mhc2_vectra_data/object_data/stelow_slides/full_image_set/panel3/summary_data.csv')
metadata_path = Path('/project/Dolatshahi_Lab/Gabe_Hanson/stelow_moore/vectra_stelow_slide_metadata.csv')

panel3   = pd.read_csv(panel3_path)
metadata = pd.read_csv(metadata_path)

# extract ROI and patient identifiers from image location string
panel3['ROI']       = panel3['Image Location'].str.extract(r'(S\d{1,2} LUNG.*?\[.*?\])')
panel3['PatientID'] = panel3['Image Location'].str.extract(r'(S\d{1,2} LUNG)')

print(f"Panel 3 ROIs loaded: {len(panel3):,}")
print(f"Patients: {panel3['PatientID'].nunique()}")

In [ ]:
# MHC II collapsed totals
panel3['tumor: PanCK+MHCII+ Cells'] = (
    panel3['tumor: PanCK+MHCI-MHCII+ Cells'] +
    panel3['tumor: PanCK+MHCI+MHCII+ Cells']
)
panel3['tumor: PanCK+MHCII- Cells'] = (
    panel3['tumor: PanCK+MHCI+MHCII- Cells'] +
    panel3['tumor: PanCK+MHCI-MHCII- Cells']
)

## Merge metadata and filter to LUAD

Clinical metadata is merged on patient code. Analysis is restricted to
adenocarcinoma (HIST == 'Adenocarcinoma'). Key metadata columns retained
for downstream use include histology, stage, sex, age, smoking history,
PD-L1 score, and MHC I pathology score (Stelow MHC).

In [ ]:
# match PatientID format to metadata Code column
panel3['Code'] = panel3['PatientID'].str.replace(' LUNG', '', regex=False)

metadata_cols = [
    'Code', 'HIST', 'SEX', 'AGE', 'STAGE', 'SMOKE_HX', 'SMOKE_PY',
    'PD-L1 Score', 'Stelow MHC', 'RACE', 'ETHN'
]
combined = panel3.merge(metadata[metadata_cols], on='Code', how='left')

# filter to LUAD
luad = combined.loc[combined['HIST'] == 'Adenocarcinoma'].copy()

print(f"LUAD ROIs: {len(luad):,}")
print(f"LUAD patients: {luad['PatientID'].nunique()}")
print(luad['PatientID'].unique())

In [ ]:
# compute per-patient PanCK+MHCII+ fraction
patient_mhc2 = (
    luad.groupby('PatientID')[['tumor: PanCK+MHCII+ Cells', 'tumor: PanCK+MHCII- Cells']]
    .sum()
    .reset_index()
)
patient_mhc2['mhc2_pos_fraction'] = (
    patient_mhc2['tumor: PanCK+MHCII+ Cells'] /
    (patient_mhc2['tumor: PanCK+MHCII+ Cells'] + patient_mhc2['tumor: PanCK+MHCII- Cells'])
)
patient_mhc2 = patient_mhc2.sort_values('mhc2_pos_fraction', ascending=False)
print(patient_mhc2[['PatientID', 'mhc2_pos_fraction']].to_string(index=False))

# plot distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    range(len(patient_mhc2)),
    patient_mhc2['mhc2_pos_fraction'],
    color=[cmap_high_low[0] if f > patient_mhc2['mhc2_pos_fraction'].median() 
           else cmap_high_low[1] for f in patient_mhc2['mhc2_pos_fraction']]
)
ax.axhline(patient_mhc2['mhc2_pos_fraction'].median(), 
           color='gray', linestyle='--', lw=1.5, label='median')
ax.set_xticks(range(len(patient_mhc2)))
ax.set_xticklabels(patient_mhc2['PatientID'].str.replace(' LUNG', ''), rotation=45, ha='right')
ax.set_ylabel('PanCK+MHCII+ fraction')
ax.legend(frameon=False)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

## MHC II patient classification

Patients are classified as MHC II high or low based on the fraction of
PanCK+ tumor cells expressing MHC II. A threshold of 0.4 is used, which
corresponds to a natural break in the distribution between S21 (0.57) and
S18 (0.26). This threshold is applied to the per-patient aggregated MHC II+
fraction across all tumor ROIs.

In [ ]:
MHCII_THRESHOLD = 0.4

# compute per-patient MHC II+ fraction from aggregated tumor counts
patient_mhc2 = (
    luad.groupby('PatientID')[['tumor: PanCK+MHCII+ Cells', 'tumor: PanCK+MHCII- Cells']]
    .sum()
    .reset_index()
)
patient_mhc2['mhc2_pos_fraction'] = (
    patient_mhc2['tumor: PanCK+MHCII+ Cells'] /
    (patient_mhc2['tumor: PanCK+MHCII+ Cells'] + patient_mhc2['tumor: PanCK+MHCII- Cells'])
)
patient_mhc2['patient classification'] = np.where(
    patient_mhc2['mhc2_pos_fraction'] >= MHCII_THRESHOLD,
    'class II high', 'class II low'
)

# merge classification back to luad
luad = luad.merge(
    patient_mhc2[['PatientID', 'patient classification']],
    on='PatientID', how='left'
)

print(luad['patient classification'].value_counts())
print("\nClassification per patient:")
print(
    patient_mhc2[['PatientID', 'mhc2_pos_fraction', 'patient classification']]
    .sort_values('mhc2_pos_fraction', ascending=False)
    .to_string(index=False)
)

# save classification table
patient_mhc2.to_csv(table_out / 'stelow_patient_mhc2_classification.csv', index=False)

In [ ]:
color_map = {
    'tumor: PanCK+MHCII+ Cells': cmap_high_low[0],
    'tumor: PanCK+MHCII- Cells': cmap_high_low[1],
}
cols = list(color_map.keys())

# aggregate and normalize per patient
df = luad.dropna(subset=cols + ['PatientID']).copy()
grouped = df.groupby('PatientID')[cols].sum()
grouped_norm = grouped.div(grouped.sum(axis=1), axis=0).reset_index()
grouped_norm['PatientNum'] = grouped_norm['PatientID'].str.extract(r'S(\d+)').astype(int)
grouped_norm = grouped_norm.sort_values('mhc2_pos_fraction' if 'mhc2_pos_fraction' in grouped_norm.columns else 'PatientNum')

# sort by MHC II+ fraction descending
grouped_norm = grouped_norm.merge(
    patient_mhc2[['PatientID', 'mhc2_pos_fraction']], on='PatientID'
).sort_values('mhc2_pos_fraction', ascending=False)

x_labels = grouped_norm['PatientID'].str.replace(' LUNG', '').values
x_pos = range(len(x_labels))

fig, ax = plt.subplots(figsize=(12, 6))
bottom = None
for cell_type in reversed(cols):
    ax.bar(
        x_pos,
        grouped_norm[cell_type],
        bottom=bottom,
        label=cell_type.replace('tumor: ', '').replace(' Cells', ''),
        color=color_map[cell_type],
    )
    bottom = grouped_norm[cell_type].copy() if bottom is None else bottom + grouped_norm[cell_type]

ax.axhline(MHCII_THRESHOLD, color='gray', linestyle='--', lw=1.5, alpha=0.7)
ax.set_ylabel('Proportion')
ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels, rotation=45, ha='right')
ax.legend(title='Cell Type', bbox_to_anchor=(1.02, 1), loc='upper left')
sns.despine(ax=ax)
plt.tight_layout()
plt.savefig(fig_out / 'fig5c_stelow_panck_mhc2_stacked_bar.pdf', bbox_inches='tight')
plt.show()